<a href="https://colab.research.google.com/github/Thabanya/nlp-insight-extraction-reviews/blob/main/Yelp_nlp_insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets sentence-transformers bertopic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
df = dataset.to_pandas()
df = df.sample(20000, random_state=42)
df.head()

,label,text
177288,0,"First of all i'm not a big fan of buffet, i tr..."
238756,1,Thanks Yelp. I was looking for the words to de...
604225,2,Service was so-so. They were receiving a deliv...
2838,2,Stamoolis Brothers is one of the Strip Distric...
586957,0,I want to give a 2 stars because the service s...


In [ ]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].str.lower()

In [ ]:
print("orginalshape:", dataset)
print("smaple shape:", df.shape)

orginalshape: Dataset({
    features: ['label', 'text'],
    num_rows: 650000
})
smaple shape: (20000, 2)


In [ ]:
df.info()
df.isnull().sum()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, 177288 to 390980
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   20000 non-null  int64 
 1   text    20000 non-null  object
dtypes: int64(1), object(1)
memory usage: 468.8+ KB


,0
label,int64
text,object


In [ ]:
df["label"].value_counts()

,count
label,
2,4039
0,4035
3,4017
1,3958
4,3951


In [ ]:
from collections import Counter

words = " ".join(df["text"]).split()
Counter(words).most_common(20)

[('the', 133461),
 ('and', 84889),
 ('i', 69330),
 ('a', 68696),
 ('to', 66115),
 ('was', 48580),
 ('of', 40142),
 ('for', 30899),
 ('it', 29840),
 ('is', 29813),
 ('in', 29220),
 ('that', 24025),
 ('my', 22904),
 ('but', 21786),
 ('we', 20235),
 ('with', 20178),
 ('this', 20054),
 ('they', 19435),
 ('on', 18207),
 ('not', 17893)]

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
df["clean_text"]

,clean_text
177288,"i'm big fan buffet, tried got $50 credit stayi..."
238756,thanks yelp. looking words place meh fitting.\...
604225,service so-so. receiving delivery it. food hot...
2838,"stamoolis brothers strip district storefronts,..."
586957,want 2 stars service staff friendly good ambie...
...,...
188178,"it's beautiful appearance, reminds younger, ha..."
574411,went fusion pizza try new place. love business...
575952,"cancelled membership june, signed months train..."
237244,yikes...their standards dropped 2 years ago. d...


In [ ]:
texts = df["clean_text"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic()

topics, probs = topic_model.fit_transform(texts, embeddings)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8675,-1_food_good_place_like,"[food, good, place, like, just, service, time,...","[reading reviews place, cautiously optimistic ..."
1,0,1030,0_mexican_tacos_salsa_taco,"[mexican, tacos, salsa, taco, burrito, chips, ...",[looking decent mexican restaurant strip came ...
2,1,835,1_room_hotel_stay_rooms,"[room, hotel, stay, rooms, stayed, desk, bed, ...","[stayed wigwam 4 days \""adobe traditional suit..."
3,2,470,2_pizza_crust_pizzas_pepperoni,"[pizza, crust, pizzas, pepperoni, slice, toppi...",[great pizza great price fresh pizza wonderful...
4,3,457,3_sushi_roll_rolls_tuna,"[sushi, roll, rolls, tuna, nigiri, fish, ayce,...",[don't understand place getting positive revie...
...,...,...,...,...,...
153,152,10,152_portions_crocketts_sharedthey_portion,"[portions, crocketts, sharedthey, portion, goo...",[good food really stingy portions price- 3+ do...
154,153,10,153_gelato_scoops_flavors_nutella,"[gelato, scoops, flavors, nutella, 535, scoop,...",[camelback greenway. yummy!! best gelato phx.....
155,154,10,154_zia_music_cds_records,"[zia, music, cds, records, equipment, gc, drum...",[harmony house music great place shop music ne...
156,155,10,155_frys_computer_employees_buy,"[frys, computer, employees, buy, store, apple,...",[shopping store opened. service usually ranges...


In [ ]:
topic_model = topic_model.reduce_topics(texts)

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8675,-1_food_good_place_like,"[food, good, place, like, just, service, time,...",[review strictly location only! love applebees...
1,0,8429,0_food_good_place_like,"[food, good, place, like, just, great, its, re...",[soooo disappointed visit ever!! understand ob...
2,1,725,1_club_music_dance_show,"[club, music, dance, show, like, its, just, pl...","[simply amazing!! cirque du soleil show, boy d..."
3,2,579,2_hair_nails_nail_salon,"[hair, nails, nail, salon, cut, gel, pedicure,...",[i'd say i'm pretty disappointed one. reading ...
4,3,540,3_car_told_service_called,"[car, told, service, called, said, customer, t...","[car died, originally towed ruffing close car ..."
5,4,200,4_massage_gym_spa_class,"[massage, gym, spa, class, classes, fitness, e...",[avid spa go-er vegas favorite place pampered....
6,5,187,5_airport_flight_car_rental,"[airport, flight, car, rental, parking, airlin...",[time flying vegas... bit shock slot machines ...
7,6,150,6_car_wash_clean_job,"[car, wash, clean, job, place, cleaners, dirty...",[got car washed 3 months ago twice price car w...
8,7,141,7_doctor_dr_office_appointment,"[doctor, dr, office, appointment, insurance, t...","[you're really sick need doctor, elsewhere! ex..."
9,8,108,8_et_le_la_u00e0,"[et, le, la, u00e0, und, des, est, pour, une, ...",[pas mal du tout ce petit resto ramen dans chi...


In [ ]:
for topic in topic_info["Topic"][:8]:  # top 8 topics
    if topic == -1:
        continue

    print("\nTopic:", topic)
    print("Keywords:", topic_model.get_topic(topic))
    print("Sample Reviews:", topic_model.get_representative_docs(topic)[:2])


Topic: 0
Keywords: [('food', np.float64(0.020384665383774203)), ('good', np.float64(0.019253868617432217)), ('place', np.float64(0.018236178326115293)), ('like', np.float64(0.01565824751105195)), ('just', np.float64(0.015252802683991852)), ('great', np.float64(0.013610188860258884)), ('its', np.float64(0.012982141538690237)), ('really', np.float64(0.012641459597336984)), ('service', np.float64(0.01233051479218388)), ('time', np.float64(0.011957120991664082))]
Sample Reviews: ['soooo disappointed visit ever!! understand obsession marie callender frozen pot pies. fall winter michigan, ate day, favorite honey chicken pot pie. just heat micro did want wait hour cook oven. microwaved pie flakey bake oven, better chicken pot pie ate night marie callender\'s restaurant! \\n\\ni craving pot pie 3 decided best. walked felt like oven, heat high thought ovens broke baking food heat vents. friendly lady counter said \\"they\\" employees complaining \\"cold\\"....i employees happy, happy work envi

### Key Insights from Yelp Customer Reviews

1. Food & Restaurant Experience (Most Discussed Topic)
   - Customers frequently talk about food quality, taste, and overall dining experience.
   - This is the most dominant topic, indicating that restaurants are the primary focus of reviews.

2. Entertainment & Nightlife
   - Reviews include experiences related to clubs, music, dance shows, and nightlife activities.
   - Customers evaluate atmosphere, crowd, and overall enjoyment.

3. Beauty & Salon Services
   - Customers discuss haircuts, nail services, and salon experiences.
   - Feedback includes service quality, hygiene, and pricing.

4. Automotive Services
   - Reviews highlight issues with car repair, customer service, and communication.
   - Many complaints relate to delays and poor service handling.

5. Health & Wellness Services
   - Includes gyms, spas, massages, and fitness centers.
   - Customers focus on service quality and facilities.

6. Travel & Airport Experience
   - Reviews mention airport services, flights, and transportation.
   - Common issues include delays and service dissatisfaction.

7. Car Cleaning & Maintenance Services
   - Customers discuss car wash quality and cleanliness.
   - Complaints often relate to incomplete or poor cleaning.

In [ ]:
topic_info = topic_model.get_topic_info()

# Remove outlier topic (-1)
filtered = topic_info[topic_info["Topic"] != -1]

# Sort by count
filtered = filtered.sort_values(by="Count", ascending=False)

filtered.head()

,Topic,Count,Name,Representation,Representative_Docs
1,0,8429,0_food_good_place_like,"[food, good, place, like, just, great, its, re...",[soooo disappointed visit ever!! understand ob...
2,1,725,1_club_music_dance_show,"[club, music, dance, show, like, its, just, pl...","[simply amazing!! cirque du soleil show, boy d..."
3,2,579,2_hair_nails_nail_salon,"[hair, nails, nail, salon, cut, gel, pedicure,...",[i'd say i'm pretty disappointed one. reading ...
4,3,540,3_car_told_service_called,"[car, told, service, called, said, customer, t...","[car died, originally towed ruffing close car ..."
5,4,200,4_massage_gym_spa_class,"[massage, gym, spa, class, classes, fitness, e...",[avid spa go-er vegas favorite place pampered....


Most Discussed Topic:
Food and restaurant-related reviews dominate the dataset,
indicating that dining experiences are the most frequently discussed by customers.

In [ ]:
filtered[["Topic", "Count"]]

,Topic,Count
1,0,8429
2,1,725
3,2,579
4,3,540
5,4,200
6,5,187
7,6,150
8,7,141
9,8,108
10,9,90


In [ ]:
topic_model.visualize_barchart()

In [ ]:
topic_model.visualize_topics()

### Conclusion

This project demonstrates how unstructured customer reviews can be transformed into meaningful insights using NLP techniques such as embeddings and topic modeling.

The analysis reveals key customer concerns and dominant themes, helping businesses better understand customer feedback and improve services.